# Hallucination detection with an LLM judge

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AgrfhyL/audio_model_initial_testing/blob/main/Colab_LLMVerdict.ipynb)

**This is the project's hallucination detector.** It follows Atwany et al. (ACL Findings 2025): an
LLM reads each reference/hypothesis pair and assigns an error category, using the prompt their
paper publishes verbatim. It classifies the 20 000 hypotheses of the delta sweep — five checkpoints
× four conditions × 1000 clips — and reports a hallucination rate per checkpoint and condition.

### What counts as a hallucination

**Fabricated content and looping output both count.** That departs from Atwany et al., who file
repetition under *Oscillation Error*, a non-hallucination class, and follows Jasiński et al., whose
HALAS-based setup treats looping as a hallucination subtype ("hallucinations containing looping,
i.e. repeated phrases").

The judge gets the paper's **fine-grained prompt** (Figure 6), which labels oscillation as its own
class. The definition is therefore applied to the *stored labels* at analysis time rather than baked
into the request:

```python
HALLUCINATION_LABELS = {"Hallucination Error", "Oscillation Error"}     # section 1
```

§8 reports that rate together with each label's share, so Atwany et al.'s narrower rate — the
"Hallucination Error" column alone — is visible in the same table. Changing the definition later
re-counts the stored labels; it never needs another API run.

The coarse prompt (Figure 5) is still selectable but **cannot support this definition**: it folds
oscillation into "Non-Hallucination Error", and once a coarse run is done nothing can separate a
loop from a misheard word. Section 3 refuses to build a run whose prompt lacks a label that
`HALLUCINATION_LABELS` names, so that mistake cannot be paid for.

If `halluc_taxonomy.csv` happens to be on Drive, sections 9–11 add a comparison against the
thresholded text taxonomy. It is optional; without it those parts are skipped.

### The judge, and the knob

**`gpt-5.6-luna`** — a July 2026 nano-tier model built for high-volume classification at low cost
per token.

**`REASONING_EFFORT` in section 1 is the knob to turn.** Valid values are `none`, `low`, `medium`
(the API default), `high`, `xhigh`, `max`. `none` ships because it matches the paper — their §A.4.1
explicitly avoids chain-of-thought "that could introduce errors" — and because reasoning tokens bill
at the **output** rate on a job whose answer is a dozen tokens long. Section 4 projects the cost of
each level before anything is spent.

**Every setting that can change a verdict gets its own run folder on Drive.** Turning the knob starts
a new run beside the old one rather than overwriting it, so two effort levels can be compared later
without paying for either again.

### Pay once, analyse forever

**The raw API responses are the record of the run.** GPT-5 family models reject `temperature`, so
the greedy decoding Atwany et al.'s §4.3 specifies is unavailable and a re-run would not reproduce
these verdicts — which is also why a re-run is never needed. Section 6 writes every response to
Drive byte-for-byte before anything parses it, and everything from section 7 on is re-derived from
those files, for free, in any runtime.

Downloading them promptly is not optional: **OpenAI deletes a batch's output file 30 days after the
batch completes.**

### Licensing: a deliberate decision, recorded here

Every other notebook in this project keeps TIMIT reference text off the wire. **This one breaks that
rule knowingly**: classifying a hypothesis against its reference requires sending both to a
third-party API, and the full-grid run transmits all 1000 references. That was an explicit choice,
recorded in each run's `run.json` so a later reader sees the decision rather than inferring it.

TIMIT is LDC93S1 — licensed, not redistributable. Confirm your LDC terms permit this before running
section 5. Nothing leaves the machine until that cell executes.

## How to run it: three phases

| phase | run | needs | touches the API |
|---|---|---|---|
| **submit** | §1 → §5 | API key, TIMIT `.TXT`, the delta sweep's files | §5 — **the only billed call** |
| **collect** | §1 → §6 | API key | polling and downloads, not billed |
| **analyse** | §1 → §7 onward | Drive files only | **never** — no key, no `openai`, no network |

Batches take minutes to hours, so these are usually three separate sessions. Every phase starts from
§1 alone: it holds the knobs and pure helpers, and imports nothing that needs a key.

**Re-running a spending cell never pays twice.** §5 submits only rows that have no verdict yet and are
not already in flight, so after a partial failure it sends just the failed rows. A batch created in
the instant before a disconnect — too late to be recorded on Drive — is recognised by the metadata it
was tagged with and adopted rather than resubmitted.

### What lands on Drive

```
MyDrive/NAACL/llm_runs/<judge>.<grain>.effort-<level>/
  run.json                    config, prompt, input digests, licensing decision, every batch
  index.csv                   custom_id -> (model, path, offset_s, timestamps); identifiers only
  requests_000.jsonl ...      exactly the bytes uploaded -- CARRIES REFERENCE TEXT, Drive-only
  batch_000.json ...          the Batch object at completion, including its token usage
  output_000.jsonl ...        raw responses, byte-for-byte as downloaded
  errors_000.jsonl ...        failed requests, when there are any
  verdict_per_utterance.csv   derived by §10: label, counted flag, usage -- no text, git-safe
  provenance.json             derived by §10: rates, label counts, realised cost, digests
```

Every raw file's SHA-256 is recorded in `run.json` at download time and checked again on every read,
so a file truncated in a Drive sync fails loudly rather than becoming a wrong HER. `llm_runs/` is
gitignored: the request files carry TIMIT references.

## 1. Config — the cell every phase starts from

No key, no network, no TIMIT. Locally it reads `$NAACL_DATA`, else `data/`, else the working
directory, so a run folder downloaded from Drive analyses on a laptop exactly as it does on Colab.

In [ ]:
import collections, csv, datetime, glob, hashlib, io, itertools, json, math, os, platform, sys, time
import numpy as np

try:                                          # Colab: everything lives on Drive
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT, ON_COLAB = "/content/drive/MyDrive/NAACL", True
except ImportError:                           # local: the same files, downloaded from Drive
    DRIVE_ROOT = os.environ.get("NAACL_DATA") or ("data" if os.path.isdir("data") else ".")
    ON_COLAB = False

# ---------------------------------------------------------------------------------------
#  THE KNOB.  Valid: "none" | "low" | "medium" (API default) | "high" | "xhigh" | "max"
#
#  Each value is its own run folder on Drive: changing it starts a new run and never touches
#  the results of an earlier one. Section 4 prices each level before you spend.
# ---------------------------------------------------------------------------------------
REASONING_EFFORT = "none"

JUDGE       = "gpt-5.6-luna"     # "gpt-4o-mini" gives a separate, greedy (reproducible) run
GRAIN       = "fine"             # "fine" = their Figure 6, labels loops separately (default)
                                 # "coarse" = Figure 5, folds loops into Non-Hallucination Error
CHUNK       = 10000              # requests per batch; ceilings are 50 000 requests / 200 MB
TEMPERATURE = 0.0                # their SS4.3 greedy decoding -- non-reasoning judges only

PRICE = {"gpt-5.6-luna": {"in": 0.20,  "out": 1.20},      # USD / 1M tokens, uncached
         "gpt-4o-mini":  {"in": 0.150, "out": 0.600}}
BATCH_DISCOUNT = 0.50

# What counts as a hallucination. Applied to the STORED labels from section 7 on, so changing it
# re-counts for free and never needs a new run. Atwany et al. count "Hallucination Error" alone;
# looping ("Oscillation Error") is counted too, following Jasinski et al.
HALLUCINATION_LABELS = {"Hallucination Error", "Oscillation Error"}

REASONING_FAMILIES = ("gpt-5", "gpt-6", "o1", "o3", "o4")


def is_reasoning(judge):
    return judge.startswith(REASONING_FAMILIES)


assert REASONING_EFFORT in ("none", "low", "medium", "high", "xhigh", "max"), REASONING_EFFORT
assert GRAIN in ("coarse", "fine"), GRAIN

# `max_completion_tokens` caps reasoning AND the answer together, so a ceiling sized for a
# twelve-token label starves any other effort. A ceiling, not a reservation: unused headroom
# is never billed.
MAX_TOKENS = 8192 if is_reasoning(JUDGE) and REASONING_EFFORT != "none" else 64

# one folder per configuration that can change a verdict
RUN = f"{JUDGE}.{GRAIN}." + (f"effort-{REASONING_EFFORT}" if is_reasoning(JUDGE) else "greedy")
RUNS_ROOT = os.path.join(DRIVE_ROOT, "llm_runs")
RUN_DIR   = os.path.join(RUNS_ROOT, RUN)

FULL_CSV = os.path.join(DRIVE_ROOT, "delta_results_full.csv")    # hypotheses       (submit)
PROV_IN  = os.path.join(DRIVE_ROOT, "delta_provenance.json")     # reference_digest (submit)
TAX_CSV  = os.path.join(DRIVE_ROOT, "halluc_taxonomy.csv")       # optional comparison

MODELS   = ["tiny", "base", "small", "medium", "large-v3"]
CONDS    = [(5, "on"), (5, "off"), (25, "on"), (25, "off")]
HEAD     = (25, "on")

print(f"data   {os.path.abspath(DRIVE_ROOT)}")
print(f"run    {RUN}   (max completion tokens {MAX_TOKENS})")
print(f"counts as hallucination: {', '.join(sorted(HALLUCINATION_LABELS))}")
print(f"folder {'exists' if os.path.isdir(RUN_DIR) else 'not created yet'}: {RUN_DIR}")

### Run-folder helpers

Pure functions over files — shared by all three phases, so the parser that decides which rows still
need a verdict in §5 is the same one that feeds the analysis in §7.

`write_bytes` re-reads what it wrote. Drive is a FUSE mount: a write that returns can still be sitting
in a buffer when the runtime dies, so a file is trusted only once its digest has been read back from
the mount.

`connect()` is the only route to the API, and only §5 and §6 call it. It installs `openai` and asks
for the key on first use, which is what keeps the analysis phase free of both.

In [ ]:
def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()


def write_bytes(path, data):
    """Write, fsync, and prove the bytes reached the mount by reading them back."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())
    digest = sha256_bytes(data)
    assert sha256_file(path) == digest, f"read-back mismatch after writing {path}"
    return digest


def write_json(path, obj):
    return write_bytes(path, json.dumps(obj, indent=1, default=str).encode())


def now():
    return datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")


def rows_tag(rows):
    """Short digest of a batch's row set -- tagged onto the batch so it can be recognised later."""
    return sha256_bytes(",".join(map(str, rows)).encode())[:32]


def run_record(run):
    p = os.path.join(RUNS_ROOT, run, "run.json")
    return json.load(open(p)) if os.path.exists(p) else None


def parse_run(run):
    """Every stored response of one run -> ({row index: record}, stats). Reads Drive only.

    Each raw file is checked against the digest recorded when it was downloaded. A row that
    appears in several batches (a retry) keeps a usable verdict over a failure, and a later
    batch over an earlier one. `billed` sums usage over every response, retried ones included,
    because those tokens were paid for too."""
    rec = run_record(run)
    assert rec is not None, f"no run.json for {run}"
    labels = set(rec["config"]["labels"])
    out, billed, dupes = {}, collections.Counter(), 0
    for b in rec["batches"]:
        if "collected_at" not in b:
            continue                                        # still in flight
        for kind in ("output", "errors"):
            if not b.get(kind):
                continue
            path = os.path.join(RUNS_ROOT, run, b[kind])
            assert sha256_file(path) == b[f"{kind}_sha256"], f"{path} differs from its download"
            for line in open(path):
                if not line.strip():
                    continue
                o    = json.loads(line)
                resp = o.get("response") or {}
                body = resp.get("body") or {}
                r = {"batch": b["k"], "status": "error", "label": "", "error_code": "",
                     "finish_reason": "", "response_model": "", "system_fingerprint": "",
                     "prompt_tokens": 0, "cached_tokens": 0,
                     "completion_tokens": 0, "reasoning_tokens": 0}
                if o.get("error") or resp.get("status_code") != 200 or not body.get("choices"):
                    err = o.get("error") or body.get("error") or {}
                    r["error_code"] = str(err.get("code") or resp.get("status_code") or "unknown")
                else:
                    ch, u = body["choices"][0], body.get("usage") or {}
                    msg = ch.get("message") or {}
                    r.update(
                        finish_reason=ch.get("finish_reason") or "",
                        response_model=body.get("model") or "",
                        system_fingerprint=body.get("system_fingerprint") or "",
                        prompt_tokens=u.get("prompt_tokens", 0),
                        cached_tokens=(u.get("prompt_tokens_details") or {}).get("cached_tokens", 0),
                        completion_tokens=u.get("completion_tokens", 0),
                        reasoning_tokens=(u.get("completion_tokens_details") or {}).get(
                            "reasoning_tokens", 0))
                    content = (msg.get("content") or "").strip()
                    if msg.get("refusal"):
                        r["status"] = "refusal"
                    elif not content:
                        r["status"] = "truncated"       # reasoning spent the whole budget
                    else:
                        try:
                            label = json.loads(content).get("label")
                        except (ValueError, AttributeError):
                            label = None
                        if label in labels:
                            r["status"], r["label"] = "ok", label
                        else:
                            r["status"] = "invalid"
                for k in ("prompt_tokens", "cached_tokens", "completion_tokens", "reasoning_tokens"):
                    billed[k] += r[k]
                i = int(o["custom_id"][1:])
                if i in out:
                    dupes += 1
                    if out[i]["status"] == "ok" and r["status"] != "ok":
                        continue
                out[i] = r
    return out, {"billed": dict(billed), "duplicates": dupes}


_CLIENT = None


def connect():
    """The OpenAI client, created on first use. Only sections 5 and 6 call this."""
    global _CLIENT
    if _CLIENT is None:
        try:
            import openai
        except ImportError:
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
            import openai
        if not os.environ.get("OPENAI_API_KEY"):
            try:                                            # Colab secret, if granted
                from google.colab import userdata
                os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
            except Exception:
                import getpass
                os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")
        _CLIENT = openai.OpenAI()
    return _CLIENT


def runs_on_drive():
    if not os.path.isdir(RUNS_ROOT):
        return []
    return sorted(d for d in os.listdir(RUNS_ROOT)
                  if os.path.exists(os.path.join(RUNS_ROOT, d, "run.json")))


print("runs on Drive:", ", ".join(runs_on_drive()) or "none yet")

## 2. Load and verify — submit phase

Same integrity checks the taxonomy notebook runs, for the same reason: a classification is only
meaningful against the corpus that produced it. References are rebuilt from TIMIT and hashed against
the `reference_digest` the delta sweep pinned — on a mismatch the LLM would be comparing hypotheses
against the wrong sentences and every verdict would be noise.

Only the submit phase needs this. Collection and analysis never read TIMIT or the hypotheses again:
§5 writes `index.csv`, which carries every row's identity into the run folder.

In [ ]:
for p in (FULL_CSV, PROV_IN):
    assert os.path.exists(p), f"missing input: {p}"
INPUT_DIGESTS = {os.path.basename(p): sha256_file(p) for p in (FULL_CSV, PROV_IN)}

full    = list(csv.DictReader(open(FULL_CSV, newline="")))
prov_in = json.load(open(PROV_IN))

cells = collections.Counter((r["model"], int(r["offset_s"]), r["timestamps"]) for r in full)
assert set(cells) == {(m, o, t) for m in MODELS for o, t in CONDS}, "grid has holes"
assert set(cells.values()) == {1000}, sorted(set(cells.values()))
assert sum("<|" in r["text"] for r in full) == 0, "special tokens leaked into text"

KEYS = [(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]) for r in full]
assert len(set(KEYS)) == len(KEYS), "duplicate (model, path, offset, arm) rows"

# --- references, verified against the delta sweep's pin --------------------------------
cands = [d for d in glob.glob(os.path.join(DRIVE_ROOT, "timit", "**", "TEST"), recursive=True)
         if os.path.isdir(os.path.join(d, "DR1"))]
assert cands, f"no TIMIT TEST/DR1 found under {DRIVE_ROOT}/timit"
TIMIT_TEST = sorted(cands, key=len)[0]

if not os.path.exists("corpus_digests.json"):
    !wget -q https://raw.githubusercontent.com/AgrfhyL/audio_model_initial_testing/main/corpus_digests.json
FILES = json.load(open("corpus_digests.json"))["files"][0:1000]

!pip -q install openai-whisper
from whisper.normalizers import EnglishTextNormalizer          # noqa: E402
normalizer = EnglishTextNormalizer()


def load_reference(wav_path):
    """TIMIT .TXT is `<start_sample> <end_sample> <sentence>` -- the integers are not words."""
    with open(wav_path[:-4] + ".TXT") as f:
        return f.read().strip().split(None, 2)[2]


REF     = {r["path"]: normalizer(load_reference(os.path.join(TIMIT_TEST, r["path"])))
           for r in FILES}
rebuilt = sha256_bytes("\n".join(f"{r['path']}\t{REF[r['path']]}" for r in FILES).encode())

# Colab_DeltaSweep.ipynb records this as provenance["corpus"]["reference_digest_sha256"]
pinned = (prov_in.get("corpus") or {}).get("reference_digest_sha256")
assert pinned, ("delta_provenance.json has no corpus.reference_digest_sha256, so the references "
                "cannot be verified -- is this the provenance file Colab_DeltaSweep.ipynb wrote?")
assert rebuilt == pinned, (
    f"reference digest mismatch (rebuilt {rebuilt[:16]}..., pinned {pinned[:16]}...): this TIMIT "
    f"copy or text normalizer differs from the one experiment A used, so every verdict would "
    f"score against the wrong sentences")
assert {k[1] for k in KEYS} <= set(REF), "hypotheses for clips outside the pinned corpus"

HYP = [normalizer(r["text"]) for r in full]          # normalized once, used by sections 4 and 5

print(f"{len(full)} hypotheses | {len(REF)} references | digest matches the delta sweep's pin")
for k, v in INPUT_DIGESTS.items():
    print(f"  {k:<28} sha256 {v[:16]}...")

## 3. The prompt, and the deliberate deviations

The instruction text below is Atwany et al.'s prompt, transcribed verbatim from the paper including
every example: **Figure 6 (fine-grained, five labels)** by default, Figure 5 (coarse, three labels)
when `GRAIN = "coarse"`. Instructions go in the `system` message and the pair in the `user` message;
the paper presents one block ending with an `Input:` template, and the rendered text and its order
are unchanged. The full prompt is stored in each run's `run.json`, so a run folder documents itself.

**Why fine-grained.** Fine-grained output keeps *Oscillation Error* apart from phonetic and language
errors, which is what lets section 1's `HALLUCINATION_LABELS` count loops as hallucinations. The
coarse prompt merges them into "Non-Hallucination Error" at classification time, irrecoverably, so
the cell below refuses a grain whose labels cannot express the chosen definition.

**Deviation 1: structured outputs replace "produce only the classification."** The paper constrains
the output by asking for it in prose; `response_format` with a strict `json_schema` constrains it by
construction. That removes the parse step and forecloses what a prose instruction cannot — a preamble
before the label, or a label spelled differently from the canonical strings. Their own §A.4.1 says
they restricted output by prompt design anyway; same intent, stronger mechanism.

**Deviation 2: no greedy decoding.** GPT-5 family models reject `temperature` outright, so the
reproducibility their §4.3 buys is unavailable. That is why the raw responses are kept: they are the
record, not something to regenerate. Setting `JUDGE = "gpt-4o-mini"` gives a separate, greedy run.

**Not a deviation: counting loops as hallucinations.** The prompt and its labels are the paper's; only
the tally differs, and §8 prints Atwany et al.'s own tally beside it.

`body_for` emits two request shapes, and mixing them is a **400, not a warning** — reasoning models
reject `temperature` and require `max_completion_tokens` where the older chat models want
`max_tokens`. The dispatch is a prefix match on the model family.

**`reasoning_effort="none"` is the faithful setting, not just the cheap one.** Their §A.4.1 says the
framework "restricts GPT-4o's output to only the final classification decision, explicitly avoiding
chain-of-thought generation that could introduce errors." Reasoning off is that decision expressed
through a parameter instead of a prompt.

`CONFIG` collects everything that can change a verdict. §5 refuses to add batches to a run folder whose
recorded `CONFIG` differs, so one folder can never mix two prompts or two effort levels. The counting
definition is deliberately **not** in `CONFIG`: it changes no verdict.

In [ ]:
COARSE_LABELS = ["Hallucination Error", "Non-Hallucination Error", "No Error"]
FINE_LABELS   = ["Phonetic Error", "Oscillation Error", "Hallucination Error",
                 "Language Error", "No Error"]

COARSE_SYSTEM = """You are a classifier trained to detect and categorize specific transcription \
errors produced by a speech recognition system. The possible categories are:
1. Hallucination Error: The output contains fabricated, contradictory, or invented information \
that is not supported by the ground truth. This includes: - Fabricated Content: Words or phrases \
entirely absent in the ground truth. - Meaningful Contradictions: Significant changes in the \
meaning from the ground truth. - Invented Context: Introduction of details or context not present \
in the ground truth. - Note: These errors involve fabrication of new information or significant \
distortion of meaning, beyond grammatical or structural mistakes.
2. Non-Hallucination Error: Errors that do not involve fabrication or significant contradictions \
of the ground truth. These include: - Phonetic Errors: Substitutions of phonetically similar words \
or minor pronunciation differences. - Structural or Language Errors: Grammatical, syntactic, or \
structural issues that make the text incoherent or incorrect (e.g., incorrect verb tenses, \
subject-verb agreement problems, omissions, or insertions). - Oscillation Errors: Repetitive, \
nonsensical patterns or sounds that do not convey linguistic meaning (e.g., "ay ay ay ay"). \
- Other Non-Hallucination Errors: Errors that do not fit the above subcategories but are not \
hallucinations.
3. No Error: The generated output conveys the same meaning as the ground truth, even if the \
phrasing, grammar, or structure differs. Minor differences in wording, phrasing, or grammar that \
do not alter the intended meaning are acceptable.

Input Format:
Ground Truth: The original, accurate text provided.
Generated Output: The text produced by the speech recognition system.

Output Format: Classify the input text pairs into one of the following:
Non-Hallucination Error
Hallucination Error
No Error

Examples:
Example 1:
Ground Truth: "A millimeter roughly equals one twenty-fifth of an inch."
Generated Output: "Miller made her roughly one twenty-fifths of an inch."
Output: Non-Hallucination Error

Example 2:
Ground Truth: "Indeed, ah!"
Generated Output: "Ay ay indeed ay ay ay ay ay ay."
Output: Non-Hallucination Error

Example 3:
Ground Truth: "Captain Lake did not look at all like a London dandy now."
Generated Output: "Will you let Annabel ask her if she sees what it is you hold in your arms \
again?"
Output: Hallucination Error

Example 4:
Ground Truth: "The patient was advised to take paracetamol for fever and rest for two days."
Generated Output: "The patient was advised to take amoxicillin for fever and undergo surgery \
immediately."
Output: Hallucination Error

Example 5:
Ground Truth: "I need to book a flight to New York."
Generated Output: "I need to book ticket to New York."
Output: No Error

Example 6:
Ground Truth: "She went to the store yesterday."
Generated Output: "She went to the shop yesterday."
Output: No Error

Instruction: You must produce only the classification as the output. Do not include explanations, \
reasoning, or additional information."""

FINE_SYSTEM = """You are a classifier trained to detect and categorize specific transcription \
errors produced by a speech recognition system. The possible categories are:
1. Phonetic Error: The output contains substitutions of phonetically similar words that do not \
match the ground truth and do not introduce broader grammatical or structural issues. These errors \
typically involve misrecognition of similar-sounding words or minor pronunciation differences.
2. Oscillation Error: The output includes repetitive, nonsensical patterns or sounds that do not \
convey linguistic meaning (e.g., "ay ay ay ay").
3. Hallucination Error: The output contains fabricated, contradictory, or invented information \
that is not supported by the ground truth. This includes: - Fabricated Content: Words or phrases \
entirely absent in the ground truth. - Meaningful Contradictions: Significant changes in the \
meaning from the ground truth. - Invented Context: Introduction of details or context not present \
in the ground truth. - Note: These errors involve fabrication of new information or significant \
distortion of meaning, beyond grammatical or structural mistakes.
4. Language Error: The output includes grammatical, syntactic, or structural issues that make the \
text incoherent or linguistically incorrect. This category encompasses errors such as: - Incorrect \
verb tenses or subject-verb agreement problems. - Sentence fragments or incomplete structures. \
- Omissions or insertions of words that do not fabricate new context. - Incomplete sentences or \
phrases that do not convey the intended meaning as ground truth. - Note: Incomplete sentences or \
phrases are classified as Language Errors only when they do not fabricate new meaning or deviate \
from the intent of the ground truth.
5. No Error: The generated output conveys the same meaning as the ground truth, even if the \
phrasing, grammar, or structure differs. Minor differences in wording, phrasing, punctuation, or \
casing that do not alter the intended meaning are not considered errors. - Note: Minor omissions, \
such as missing articles, are acceptable as long as they do not change the meaning of the ground \
truth.

Input Format:
Ground Truth: The original, accurate text provided.
Generated Output: The text produced by the speech recognition system.

Output Format: Classify the input text pairs into one of the following:
Phonetic Error
Oscillation Error
Hallucination Error
Language Error
No Error

Examples:
Example 1:
Ground Truth: "A millimeter roughly equals one twenty-fifth of an inch."
Generated Output: "Miller made her roughly one twenty-fifths of an inch."
Output: Phonetic Error

Example 2:
Ground Truth: "I will go to New York City!"
Generated Output: "Ay ay ay ay ay ay ay ay."
Output: Oscillation Error

Example 3:
Ground Truth: "Captain Lake did not look at all like a London dandy now."
Generated Output: "Will you let Annabel ask her if she sees what it is you hold in your arms \
again?"
Output: Hallucination Error

Example 4:
Ground Truth: "The cat is chasing the mouse."
Generated Output: "The cat chased by the mouse."
Output: Language Error

Example 5:
Ground Truth: "I need to book a flight to New York."
Generated Output: "I need to book ticket to New York."
Output: No Error

Your Task: Classify the input into one of the five categories.
Instruction: You must produce only the classification as the output. Do not include explanations, \
reasoning, or additional information."""

SYSTEM = COARSE_SYSTEM if GRAIN == "coarse" else FINE_SYSTEM
LABELS = COARSE_LABELS if GRAIN == "coarse" else FINE_LABELS

# strict mode requires additionalProperties:false and every property listed in required
SCHEMA = {"type": "object",
          "properties": {"label": {"type": "string", "enum": LABELS}},
          "required": ["label"], "additionalProperties": False}
RESPONSE_FORMAT = {"type": "json_schema",
                   "json_schema": {"name": "verdict", "strict": True, "schema": SCHEMA}}

unsupported = set(HALLUCINATION_LABELS) - set(LABELS)
assert not unsupported, (
    f"the {GRAIN} prompt has no {sorted(unsupported)} label, so HALLUCINATION_LABELS cannot be "
    f"counted from its verdicts -- and a finished run cannot be relabelled. Use GRAIN = 'fine', or "
    f"drop those labels from HALLUCINATION_LABELS deliberately.")
USER_TEMPLATE = 'Ground Truth: "{ref}"\nGenerated Output: "{hyp}"'


def user_turn(ref, hyp):
    return USER_TEMPLATE.format(ref=ref, hyp=hyp)


def body_for(ref, hyp):
    """One /v1/chat/completions body for JUDGE. The two families take different parameters and
    mixing them is a 400: reasoning models reject `temperature` and require
    `max_completion_tokens` where the older chat models want `max_tokens`."""
    b = {
        "model": JUDGE,
        "messages": [{"role": "system", "content": SYSTEM},
                     {"role": "user",   "content": user_turn(ref, hyp)}],
        "response_format": RESPONSE_FORMAT,
    }
    if is_reasoning(JUDGE):
        b["max_completion_tokens"] = MAX_TOKENS
        b["reasoning_effort"] = REASONING_EFFORT   # "none": no chain of thought, as they specify
    else:
        b["max_tokens"] = MAX_TOKENS
        b["temperature"] = TEMPERATURE             # the paper's greedy decoding, verbatim
    return b


# everything that can change a verdict; section 5 refuses to mix two of these in one folder
CONFIG = {
    "judge": JUDGE, "grain": GRAIN, "labels": LABELS,
    "reasoning_effort": REASONING_EFFORT if is_reasoning(JUDGE) else None,
    "temperature": None if is_reasoning(JUDGE) else TEMPERATURE,
    "system_sha256": sha256_bytes(SYSTEM.encode()),
    "user_template": USER_TEMPLATE,
    "response_format": RESPONSE_FORMAT,
    "endpoint": "/v1/chat/completions",
}

how = (f"reasoning_effort={REASONING_EFFORT}, no temperature (rejected by this family)"
       if is_reasoning(JUDGE) else f"temperature={TEMPERATURE} (greedy, reproducible)")
print(f"{JUDGE} | grain={GRAIN} | {how}")
print(f"{len(LABELS)} labels: {', '.join(LABELS)} | system block {len(SYSTEM)} chars")
print(f"counted as hallucination at analysis time: {', '.join(sorted(HALLUCINATION_LABELS))}")

## 4. Measure before spending

`gpt-5.6-luna` bills at **\$0.20 / \$1.20 per 1M** tokens, halved by the Batch API. OpenAI's caching
is automatic and only engages on prefixes of **1024 tokens or more** — the cell below measures the
instruction block and reports whether it clears that bar. Neither prompt does — Figure 6 is 705 tokens,
Figure 5 654 — so the projection assumes no caching.

The per-row part is measured over **every row**, not one example: the user turn carries the reference
and the hypothesis, and a looping hypothesis is several times longer than a clean one. Chat framing
and the injected schema add a few dozen tokens more, estimated as `OVERHEAD`; §7 reports the real
`prompt_tokens` afterwards.

**Output is the line that moves.** Reasoning tokens bill at the output rate, so the cost is decided by
`REASONING_EFFORT`. The table is a **sensitivity sweep, not a point estimate**: there is no measured
reasoning-token figure for this prompt at `low`/`medium`/`high`, and inventing one would be worse
than showing the shape. §7 prints the observed per-row figure after a run.

Worth knowing while reading their paper: Atwany et al.'s App. A.4.1 rate card quotes
"\$0.075 per million input tokens" — the *cached* rate — alongside "with most input tokens cached",
so their \$78-per-million-segments figure assumes a near-perfect hit rate. Padding the prompt past
1024 tokens to get caching is deliberately **not** done: the saving is a fraction of a dollar and it
would mean no longer running the paper's prompt.

In [ ]:
!pip -q install tiktoken
import tiktoken                                              # noqa: E402

CACHE_MIN, OVERHEAD, OUT_TOK = 1024, 30, 12
enc = tiktoken.get_encoding("o200k_base")                     # the gpt-4o / gpt-5 tokenizer family

sys_tok = len(enc.encode(SYSTEM))
var_tok = np.array([len(enc.encode(user_turn(REF[k[1]], HYP[i]))) for i, k in enumerate(KEYS)])
N, in_tok = len(KEYS), sys_tok + var_tok.mean() + OVERHEAD

print(f"system block {sys_tok} tokens | user turn mean {var_tok.mean():.1f}, "
      f"p99 {np.percentile(var_tok, 99):.0f}, max {var_tok.max()} | framing ~{OVERHEAD} | N = {N}")
print(f"automatic caching engages at {CACHE_MIN} tokens: "
      f"{'YES, prefix qualifies' if sys_tok >= CACHE_MIN else 'no, prefix is below the bar'}")
print(f"REASONING_EFFORT = {REASONING_EFFORT!r}  (max completion tokens {MAX_TOKENS})\n")

p = PRICE.get(JUDGE)
assert p is not None, f"no rate card for {JUDGE}: add it to PRICE in section 1"
SWEEP = [0, 50, 200, 500, 1000]                                # reasoning tokens per row
cin = N * in_tok / 1e6 * p["in"] * BATCH_DISCOUNT
hdr = f"{'judge':>15} {'input $':>9} " + " ".join(f"{f'+{r} reas':>10}" for r in SWEEP)
print("batch-discounted total ($), by reasoning tokens per row\n")
print(hdr + "\n" + "-" * len(hdr))
print(f"{JUDGE:>15} {cin:>9.2f} " + " ".join(
    f"{cin + N * (OUT_TOK + r) / 1e6 * p['out'] * BATCH_DISCOUNT:>10.2f}" for r in SWEEP))

print("\nThe +0 column is what effort 'none' should cost. The rest is a sweep, not a prediction.")
print("Bound on input: every hypothesis at Whisper's 224-token decode ceiling would add roughly "
      f"${N * (224 - var_tok.mean()) / 1e6 * p['in'] * BATCH_DISCOUNT:.2f}.")

## 5. Submit — the cell that sends reference text off this machine

**This is the licensing boundary and the only billed call.** Running it transmits TIMIT reference
transcripts, plus hypotheses, to the OpenAI API. Nothing above this point has left the runtime.

What it does, in order, and why each step is where it is:

1. **Opens or creates the run folder** and refuses to continue if the folder's recorded `CONFIG` or
   input digests differ from this session's — one folder never mixes two prompts, two effort levels,
   or two versions of the hypotheses.
2. **Writes `index.csv`** (row → model, path, offset, arm). From here on nothing needs TIMIT or the
   hypotheses to know which verdict belongs to which utterance.
3. **Works out which rows to send**: every row without a usable verdict in the stored responses and
   not already in a batch that is still running. On a first run that is all 20 000; after a partial
   failure it is only the failures.
4. **Writes each request file to Drive, then uploads that file.** What was sent is on record exactly.
5. **Tags each batch with `metadata`** (run name and a digest of its rows) and records it in
   `run.json` before the next chunk. If the runtime died between creating a batch and recording it,
   the next run finds the tagged batch on the account and adopts it instead of paying for it again.

The Batch API takes a JSONL file, one call per line with `custom_id`, `method`, `url` and `body`;
ceilings are 50 000 requests and 200 MB per batch. 20 000 rows at this prompt size is roughly 66 MB,
so `CHUNK = 10000` stays inside both.

In [ ]:
CONFIRM = False        # set True to send reference + hypothesis text to the OpenAI API

assert CONFIRM, ("set CONFIRM = True to proceed. This transmits TIMIT reference transcripts "
                 "(LDC93S1, licensed) to a third-party API; confirm your LDC terms permit it.")

client = connect()
RUN_JSON = os.path.join(RUN_DIR, "run.json")
INPUTS = {**INPUT_DIGESTS, "reference_digest": rebuilt}

rec = run_record(RUN)
if rec is None:
    rec = {"run": RUN, "created_at": now(), "config": CONFIG, "system_prompt": SYSTEM,
           "price_per_1m": PRICE.get(JUDGE), "batch_discount": BATCH_DISCOUNT,
           "inputs": INPUTS,
           "packages": {"python": platform.python_version(),
                        **{m: sys.modules[m].__version__
                           for m in ("openai", "tiktoken", "numpy") if m in sys.modules}},
           "licensing": {
               "decision": "full-grid submission chosen deliberately by the author",
               "consequence": "TIMIT (LDC93S1) reference transcripts and the hypotheses were "
                              "transmitted to the OpenAI API",
               "note": "every other notebook in this project keeps reference text off the wire; "
                       "this one does not, and the trade was made knowingly"},
           "batches": []}
else:
    canon = lambda v: json.dumps(v, sort_keys=True)            # noqa: E731
    drift = [k for k in CONFIG if canon(rec["config"].get(k)) != canon(CONFIG[k])]
    assert not drift, (f"{RUN} was recorded with a different {drift}. One folder holds one "
                       f"configuration: revert the change, or give it a new run name.")
    assert rec["inputs"] == INPUTS, (f"{RUN} was submitted against different input files "
                                     f"({rec['inputs']} vs {INPUTS})")

# --- index: every row's identity, so nothing later needs TIMIT or the hypotheses ----------
buf = io.StringIO()
wr = csv.writer(buf, lineterminator="\n")
wr.writerow(["i", "model", "path", "offset_s", "timestamps"])
wr.writerows([i, *k] for i, k in enumerate(KEYS))
INDEX = os.path.join(RUN_DIR, "index.csv")
if os.path.exists(INDEX):
    assert open(INDEX, newline="").read() == buf.getvalue(), "index.csv on Drive differs from KEYS"
else:
    write_bytes(INDEX, buf.getvalue().encode())
write_json(RUN_JSON, rec)

# --- which rows still need sending ----------------------------------------------------
parsed    = parse_run(RUN)[0] if rec["batches"] else {}
ok        = {i for i, r in parsed.items() if r["status"] == "ok"}
in_flight = {i for b in rec["batches"] if "collected_at" not in b for i in b["rows"]}
todo      = [i for i in range(len(KEYS)) if i not in ok and i not in in_flight]
print(f"{RUN}: {len(ok)} verdicts stored, {len(in_flight)} in flight, {len(todo)} to submit")

# batches that exist on the account but never reached run.json (a disconnect mid-cell)
recorded = {b["batch_id"] for b in rec["batches"]}
orphans = {(b.metadata or {}).get("rows"): b for b in client.batches.list(limit=100).data
           if (b.metadata or {}).get("run") == RUN and b.id not in recorded
           and b.status != "failed"} if todo else {}

for start in range(0, len(todo), CHUNK):
    rows = todo[start:start + CHUNK]
    k, tag = len(rec["batches"]), rows_tag(rows)
    name = f"requests_{k:03d}.jsonl"
    payload = "".join(json.dumps({"custom_id": f"r{i}", "method": "POST",
                                  "url": CONFIG["endpoint"],
                                  "body": body_for(REF[KEYS[i][1]], HYP[i])}) + "\n"
                      for i in rows).encode()
    assert len(payload) < 200e6, f"{len(payload)/1e6:.0f} MB exceeds the 200 MB ceiling; lower CHUNK"
    entry = {"k": k, "n": len(rows), "rows": rows, "max_tokens": MAX_TOKENS,
             "requests": name, "requests_sha256": write_bytes(os.path.join(RUN_DIR, name), payload)}

    if tag in orphans:
        b = orphans.pop(tag)
        entry.update(batch_id=b.id, input_file_id=b.input_file_id, adopted=True,
                     submitted_at=datetime.datetime.fromtimestamp(
                         b.created_at, datetime.timezone.utc).isoformat(timespec="seconds"))
        how = "adopted an unrecorded batch -- not resubmitted"
    else:
        with open(os.path.join(RUN_DIR, name), "rb") as fh:      # upload the file on record
            up = client.files.create(file=fh, purpose="batch")
        b = client.batches.create(input_file_id=up.id, endpoint=CONFIG["endpoint"],
                                  completion_window="24h",
                                  metadata={"run": RUN, "k": str(k), "rows": tag})
        entry.update(batch_id=b.id, input_file_id=up.id, submitted_at=now())
        how = f"submitted ({len(payload)/1e6:.0f} MB)"

    rec["batches"].append(entry)
    write_json(RUN_JSON, rec)                                   # recorded before the next chunk
    print(f"  batch {k}: {len(rows)} rows -> {b.id}  {how}")

print(f"\n{len(rec['batches'])} batches on record in {RUN_JSON}")
print("Next: section 6 collects them -- in this runtime or any later one.")

## 6. Collect — run §1, then this, in any later session

Polls every batch in the run folder that has not been downloaded yet. **As each one finishes, its
files go straight to Drive** — the Batch object, the raw output, and the error file if there is one —
and each is read back and digest-checked before `run.json` records it. A disconnect mid-poll loses
nothing already downloaded; re-running the cell picks up where it stopped.

Nothing is parsed here. Parsing is §7's job and reads Drive only, which is what makes it free to
repeat.

Two Batch API details that make the obvious version of this wrong:

- **Failed requests are not in the output file.** They go to a separate `error_file_id`, so a batch
  that silently dropped rows still yields a clean-looking output file. Both are kept, and §7 counts
  every row.
- **An expired or cancelled batch still has results.** Requests that completed before the 24-hour
  window closed are billed and returned; they are downloaded like any other, and §5 resubmits only
  the rest.

`POLL_SECONDS = 0` makes a single pass instead of waiting. `DELETE_REMOTE = True` removes this run's
input and output files from OpenAI once their Drive copies are verified — optional hygiene for the
licensed text, since nothing in this notebook reads them from OpenAI again.

In [ ]:
POLL_SECONDS  = 60      # 0 = one pass: collect whatever has finished, then return
DELETE_REMOTE = False   # True = delete this run's files from OpenAI once the Drive copies verify

TERMINAL = {"completed", "failed", "expired", "cancelled"}
RUN_JSON = os.path.join(RUN_DIR, "run.json")
rec = run_record(RUN)
assert rec is not None, f"nothing submitted for {RUN}: run section 5 first"
client = connect()


def download(entry, batch):
    """Batch object + raw output + errors -> Drive, digest-checked, then recorded."""
    k = entry["k"]
    write_json(os.path.join(RUN_DIR, f"batch_{k:03d}.json"), batch.model_dump())
    for kind, fid in (("output", batch.output_file_id), ("errors", batch.error_file_id)):
        if not fid:
            continue
        data = client.files.content(fid).content          # bytes, exactly as served
        entry[kind] = f"{kind}_{k:03d}.jsonl"
        entry[f"{kind}_sha256"] = write_bytes(os.path.join(RUN_DIR, entry[kind]), data)
        entry[f"{kind}_lines"] = sum(1 for line in data.splitlines() if line.strip())
        entry[f"{kind}_file_id"] = fid
    entry["status"], entry["collected_at"] = batch.status, now()
    write_json(RUN_JSON, rec)


while True:
    for e in [e for e in rec["batches"] if "collected_at" not in e]:
        b = client.batches.retrieve(e["batch_id"])
        if b.status not in TERMINAL:
            print(f"  batch {e['k']}: {b.status} "
                  f"({b.request_counts.completed}/{b.request_counts.total})", flush=True)
            continue
        download(e, b)
        got = e.get("output_lines", 0) + e.get("errors_lines", 0)
        print(f"  batch {e['k']}: {b.status} -> {e.get('output_lines', 0)} responses, "
              f"{e.get('errors_lines', 0)} errors ({got}/{e['n']} rows) saved to Drive")
        if b.status == "failed" and b.errors:          # codes and line numbers only, no text
            print("    batch rejected:", [(x.code, x.line) for x in (b.errors.data or [])][:5])
    pending = [e for e in rec["batches"] if "collected_at" not in e]
    if not pending or not POLL_SECONDS:
        break
    time.sleep(POLL_SECONDS)

if pending:
    print(f"\n{len(pending)} batch(es) still running -- re-run this cell later; nothing is lost")
else:
    print(f"\nevery batch of {RUN} is on Drive, digests verified.")
    print("Sections 7 onward read only those files: no key, no API, no cost, any runtime.")
    if DELETE_REMOTE:
        for e in rec["batches"]:
            for fid in (e.get("input_file_id"), e.get("output_file_id"), e.get("errors_file_id")):
                if fid:
                    try:
                        client.files.delete(fid)
                    except Exception as err:                # already gone is fine
                        print(f"  could not delete {fid}: {type(err).__name__}")
        rec["remote_files_deleted_at"] = now()
        write_json(RUN_JSON, rec)
        print("remote copies deleted from OpenAI")

## 7. Parse the stored responses — the analysis phase starts here

**From this cell on nothing calls the API.** Run §1, then any of §7–§12, in a fresh runtime with no
key: everything is re-derived from the files §6 saved. That makes analysis free to repeat, and it is
why a new figure, a new table, or a **new definition of hallucination** never costs a second
classification run.

`ANALYSE` lists the runs to load. It defaults to the one configured in §1; add another run's folder
name — say the same judge at a second effort level — and every section below reports both, plus the
agreement between them.

`HALLUCINATION_LABELS` from §1 is applied here, to the stored labels. Every run is checked for it: a
run classified with a prompt that lacks one of those labels stops this cell rather than silently
under-counting.

Each row gets a status. Only `ok` carries a verdict; the rest are counted, never guessed:

| status | meaning |
|---|---|
| `ok` | a label from the fixed set |
| `truncated` | empty content — reasoning spent the whole `max_completion_tokens` budget |
| `refusal` | the model declined; structured outputs return this in a separate field |
| `invalid` | content that is not one of the labels (strict mode should make this impossible) |
| `error` | the request failed; the code is kept |
| `missing` | no response anywhere — the batch expired or was cancelled before reaching it |

Analysis needs a verdict for every row, so anything short of that stops here with the count and the
remedy: re-run §1–§5, which submits only the rows that lack one.

Speaker and dialect region come from the TIMIT path itself (`DR4/FADG0/SX289.WAV`), so nothing here
needs another notebook's output. `halluc_taxonomy.csv` is read only if it is present.

**Reasoning tokens are checked, not assumed.** At `effort="none"` more than a handful per row means
the parameter did not take effect, and the run cost more than §4 projected; this cell says so.

In [ ]:
ANALYSE = [RUN]       # add run folder names to analyse several side by side

on_drive = runs_on_drive()
missing_runs = [r for r in ANALYSE if r not in on_drive]
assert not missing_runs, f"not on Drive: {missing_runs}; available: {on_drive}"
COUNTED = set(HALLUCINATION_LABELS)

RUNREC, VERDICT, USAGE, PARSE, KEYS_ = {}, {}, {}, {}, None
hdr = (f"{'run':>34} {'ok':>6} {'trunc':>6} {'refus':>6} {'inval':>6} {'error':>6} {'miss':>6}"
       f" {'retried':>8}")
print(hdr + "\n" + "-" * len(hdr))
problems = []
for run in ANALYSE:
    rec = RUNREC[run] = run_record(run)
    lacking = COUNTED - set(rec["config"]["labels"])
    if lacking:
        problems.append(f"{run}: its {rec['config']['grain']} prompt has no {sorted(lacking)} label, "
                        f"so HALLUCINATION_LABELS cannot be counted from it")
    idx = list(csv.DictReader(open(os.path.join(RUNS_ROOT, run, "index.csv"), newline="")))
    keys = [(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]) for r in idx]
    assert [int(r["i"]) for r in idx] == list(range(len(idx))), f"{run}: index.csv out of order"
    KEYS_ = KEYS_ or keys
    assert set(keys) == set(KEYS_), f"{run} covers different rows from {ANALYSE[0]}"

    in_flight = [b["k"] for b in rec["batches"] if "collected_at" not in b]
    parsed, stats = parse_run(run)
    st = collections.Counter(r["status"] for r in parsed.values())
    st["missing"] = len(keys) - len(parsed)
    print(f"{run:>34} {st['ok']:>6} {st['truncated']:>6} {st['refusal']:>6} {st['invalid']:>6}"
          f" {st['error']:>6} {st['missing']:>6} {stats['duplicates']:>8}")
    if in_flight:
        problems.append(f"{run}: batches {in_flight} not downloaded yet -- run section 6")
    elif st["ok"] < len(keys):
        problems.append(f"{run}: {len(keys) - st['ok']} of {len(keys)} rows have no verdict -- "
                        f"re-run sections 1-5 (submits only those rows), then section 6")
    VERDICT[run] = {keys[i]: r for i, r in parsed.items()}
    USAGE[run], PARSE[run] = stats["billed"], {"status": dict(st), **stats}

assert not problems, "\n".join(problems)

print(f"\n{'run':>34} {'prompt tok':>11} {'cached':>8} {'completion':>11} {'reasoning':>10}"
      f" {'reas/row':>9} {'cost $':>7}")
for run in ANALYSE:
    u, rec = USAGE[run], RUNREC[run]
    n = len(VERDICT[run])
    price = rec.get("price_per_1m")
    cost = ((u["prompt_tokens"] * price["in"] + u["completion_tokens"] * price["out"]) / 1e6
            * rec.get("batch_discount", 1.0)) if price else float("nan")
    PARSE[run]["realised_cost_usd"] = cost
    print(f"{run:>34} {u['prompt_tokens']:>11} {u['cached_tokens']:>8} {u['completion_tokens']:>11}"
          f" {u['reasoning_tokens']:>10} {u['reasoning_tokens'] / n:>9.1f} {cost:>7.2f}")
    models = collections.Counter(r["response_model"] for r in VERDICT[run].values())
    PARSE[run]["response_models"] = dict(models)
    print(f"{'':>34} served by: {', '.join(f'{m} ({c})' for m, c in models.items())}")
    if rec["config"].get("reasoning_effort") == "none" and u["reasoning_tokens"] / n >= 5:
        print(f"{'':>34} WARNING: {u['reasoning_tokens'] / n:.0f} reasoning tokens/row despite "
              f"effort='none' -- the parameter did not take effect")

TAX, TAX_SHA = None, None
if os.path.exists(TAX_CSV):
    TAX_SHA = sha256_file(TAX_CSV)
    TAX = {(r["model"], r["path"], int(r["offset_s"]), r["timestamps"]): r
           for r in csv.DictReader(open(TAX_CSV, newline=""))}
    assert set(TAX) == set(KEYS_), "halluc_taxonomy.csv covers different rows from the runs"

SOURCES = (["taxonomy"] if TAX else []) + ANALYSE
rows = []
for key in KEYS_:
    parts = key[1].split("/")
    assert len(parts) == 3 and parts[0].startswith("DR"), f"unexpected TIMIT path {key[1]}"
    row = {"model": key[0], "path": key[1], "offset_s": key[2], "timestamps": key[3],
           "region": parts[0], "speaker": parts[1]}
    if TAX:
        row["category"], row["taxonomy"] = TAX[key]["category"], int(TAX[key]["halluc"])
    for run in ANALYSE:
        row[f"label|{run}"] = VERDICT[run][key]["label"]
        row[run] = int(row[f"label|{run}"] in COUNTED)
    rows.append(row)

BY = collections.defaultdict(list)
for r in rows:
    BY[(r["model"], r["offset_s"], r["timestamps"])].append(r)
assert all(len(BY[(m, o, t)]) == 1000 for m in MODELS for o, t in CONDS), "grid has holes"
print(f"\n{len(rows)} rows | counted as hallucination: {', '.join(sorted(COUNTED))}")
print("taxonomy comparison: " + ("included (halluc_taxonomy.csv found)" if TAX else
                                 "skipped (no halluc_taxonomy.csv on Drive)"))

## 8. Hallucination rate

The share of utterances in each cell whose label is in `HALLUCINATION_LABELS` — fabricated content
**and** looping, by default. Same denominator as Atwany et al.'s **HER** (hallucination errors over
total examples); the numerator is wider by exactly the Oscillation Error rows.

The second table breaks the rate into its labels. Its "Hallucination Error" column **is** Atwany et
al.'s HER under their own definition, so the paper-comparable number and the project's number come
from the same run and sit side by side — and the table shows how much of any positional or scale
effect is carried by loops rather than by fabrication.

Wilson intervals, and the speaker-clustered bootstrap on the headline condition, for the same reason
as everywhere else in this project: 168 speakers contribute 5–6 clips each, so utterances are not
independent.

In [ ]:
def wilson(k, n, z=1.959963985):
    if n == 0:
        return float("nan"), 0.0, 1.0
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def cluster_ci(flags, speakers, n_boot=10000, alpha=0.05, seed=0):
    f, g = np.asarray(flags, float), np.asarray(speakers)
    pool = [np.flatnonzero(g == u) for u in np.unique(g)]
    K, rng = len(pool), np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        boots[b] = f[np.concatenate([pool[j] for j in rng.integers(0, K, K)])].mean()
    return float(np.quantile(boots, alpha / 2)), float(np.quantile(boots, 1 - alpha / 2))


def ci(lo, hi):
    """Format an interval without nesting quotes inside an f-string (works on Python 3.9+)."""
    return "[" + f"{100*lo:5.2f}" + ", " + f"{100*hi:5.2f}" + "]"


SUMMARY = {}
for src in SOURCES:
    name = ("taxonomy hallucination rate (%)" if src == "taxonomy"
            else f"hallucination rate (%) -- {src}")
    print(f"\n{name}   [Wilson 95% CI]\n")
    hdr = f"{'model':>9} " + " ".join(f"{f'{o} s / ts {t}':>21}" for o, t in CONDS)
    print(hdr + "\n" + "-" * len(hdr))
    for m in MODELS:
        out = []
        for o, t in CONDS:
            rs = BY[(m, o, t)]
            p, lo, hi = wilson(sum(r[src] for r in rs), len(rs))
            s = SUMMARY[f"{src}|{m}|{o}|{t}"] = {"n": len(rs), "rate": p,
                                                 "wilson_lo": lo, "wilson_hi": hi}
            if src != "taxonomy":                  # the full label distribution, for provenance
                s["label_counts"] = dict(collections.Counter(r[f"label|{src}"] for r in rs))
            out.append(f"{100*p:5.1f} [{100*lo:4.1f},{100*hi:4.1f}]")
        print(f"{m:>9} " + " ".join(f"{c:>21}" for c in out))

for run in ANALYSE:
    labs = [l for l in RUNREC[run]["config"]["labels"] if l in COUNTED]
    print(f"\nwhat the rate is made of (% of utterances) -- {run}")
    print("the 'Hallucination Error' column alone is Atwany et al.'s HER\n")
    hdr = (f"{'model':>9} {'condition':>14} " + " ".join(f"{l:>20}" for l in labs)
           + f" {'counted':>8}")
    print(hdr + "\n" + "-" * len(hdr))
    for m in MODELS:
        for o, t in CONDS:
            s = SUMMARY[f"{run}|{m}|{o}|{t}"]
            print(f"{m:>9} {f'{o} s / ts {t}':>14} "
                  + " ".join(f"{100 * s['label_counts'].get(l, 0) / s['n']:>20.1f}" for l in labs)
                  + f" {100 * s['rate']:>8.1f}")

print(f"\nspeaker-clustered cross-check at {HEAD[0]} s / ts {HEAD[1]}\n")
hdr = f"{'source':>34} {'model':>10} {'wilson 95%':>18} {'speaker-clustered':>20}"
print(hdr + "\n" + "-" * len(hdr))
for src in SOURCES:
    for m in MODELS:
        rs = BY[(m, *HEAD)]
        s  = SUMMARY[f"{src}|{m}|{HEAD[0]}|{HEAD[1]}"]
        lo, hi = cluster_ci([r[src] for r in rs], [r["speaker"] for r in rs])
        s["cluster_lo"], s["cluster_hi"] = lo, hi
        print(f"{src:>34} {m:>10} {ci(s['wilson_lo'], s['wilson_hi']):>18} {ci(lo, hi):>20}")

## 9. Agreement — only when there is something to compare

With one run and no taxonomy file this section prints a notice and does nothing. It has work to do
in two cases:

- **Several runs in `ANALYSE`** — e.g. two effort levels. Agreement between them says how much the
  knob moves individual verdicts, separately from how much it moves the rate.
- **`halluc_taxonomy.csv` on Drive** — agreement with the thresholded text detector, plus a table of
  where they disagree by taxonomy category. Atwany et al. report human–heuristic agreement of
  **0.00** for a threshold heuristic (their human–GPT figure is 0.60), which is the reference point.

Three quantities, because agreement alone is ambiguous at low base rates: **raw agreement** (dominated
by the rows both call clean), **Cohen's kappa**, which corrects for chance, and **positive-class
Jaccard**, where the two methods actually have to decide something.

With loops counted on both sides, the taxonomy's `degenerate` rows should now mostly agree. Where they
do not, the judge has read a loop as something other than oscillation or fabrication — usually
Language Error — and those rows are worth a look.

In [ ]:
def agree(a, b):
    """Raw agreement, Cohen's kappa, and Jaccard over the positive class."""
    a, b = np.asarray(a, int), np.asarray(b, int)
    po = float((a == b).mean())
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    both, either = int((a & b).sum()), int((a | b).sum())
    return po, kappa, (both / either if either else float("nan")), both, either


AGREE = {}
if len(SOURCES) < 2:
    print("one source only -- nothing to compare. Add a run to ANALYSE, or put "
          "halluc_taxonomy.csv on Drive, to use this section.")
else:
    print(f"agreement over all {len(rows)} rows\n")
    hdr = f"{'pair':>72} {'raw':>7} {'kappa':>7} {'jaccard':>8} {'both':>6} {'either':>7}"
    print(hdr + "\n" + "-" * len(hdr))
    for x, y in itertools.combinations(SOURCES, 2):
        po, k, jac, both, either = agree([r[x] for r in rows], [r[y] for r in rows])
        AGREE[f"{x} vs {y}"] = {"raw": po, "kappa": k, "jaccard": jac,
                                "both": both, "either": either}
        print(f"{f'{x} vs {y}':>72} {po:>7.3f} {k:>7.3f} {jac:>8.3f} {both:>6} {either:>7}")

if TAX:
    print("\nAtwany et al. Table 4 for reference:")
    print("  human-human 0.71 | human-GPT 0.60 | human-Gemini 0.59 | GPT-Gemini 0.78")
    print("  human-heuristic 0.00 | GPT-heuristic 0.10 | heuristic-Gemini 0.14")
    CATS = ["faithful", "errorful", "truncated", "empty", "degenerate", "runaway", "untethered"]
    for run in ANALYSE:
        print(f"\nwhere the taxonomy and {run} disagree, by taxonomy category "
              f"({HEAD[0]} s / ts {HEAD[1]})\n")
        hdr = (f"{'taxonomy category':>18} {'n':>6} {'llm halluc':>11} {'tax halluc':>11}"
               f" {'disagree':>9}")
        print(hdr + "\n" + "-" * len(hdr))
        for cat in CATS:
            rs = [r for m in MODELS for r in BY[(m, *HEAD)] if r["category"] == cat]
            if rs:
                print(f"{cat:>18} {len(rs):>6} {sum(r[run] for r in rs):>11}"
                      f" {sum(r['taxonomy'] for r in rs):>11}"
                      f" {sum(r['taxonomy'] != r[run] for r in rs):>9}")

## 10. Write the derived results

Two files per run, both **derived** — rewritten from the raw responses every time this runs, so they
can be deleted and regenerated for free. Raw files are never touched.

`verdict_per_utterance.csv` is git-safe by construction: identifiers, labels from a fixed set, and
numbers. It keeps the **full five-way label** in `verdict` beside the `halluc` flag, so any other
definition of hallucination is one `csv` read away — as are finish reason, per-row token usage, the
exact model snapshot that answered, and which batch it came from. Taxonomy columns are added only
when `halluc_taxonomy.csv` was found.

`provenance.json` carries the run's configuration and licensing decision forward from `run.json`, and
records the counting definition, the hallucination rate with each cell's full label distribution,
agreement (when computed), status counts and the realised cost.

In [ ]:
SAFE_FIELDS = (["model", "path", "speaker", "region", "offset_s", "timestamps"]
               + (["taxonomy", "taxonomy_halluc"] if TAX else [])
               + ["verdict", "halluc", "finish_reason",
                  "prompt_tokens", "cached_tokens", "completion_tokens", "reasoning_tokens",
                  "response_model", "system_fingerprint", "batch"])
FORBIDDEN = {"text", "reference", "hypothesis", "ref", "hyp", "transcript", "content", "refusal"}
IDENT = {"model", "path", "speaker", "region", "timestamps", "taxonomy", "verdict"}
assert not (set(SAFE_FIELDS) & FORBIDDEN), "a text-bearing column leaked into SAFE_FIELDS"

for run in ANALYSE:
    rec, cfg = RUNREC[run], RUNREC[run]["config"]
    out = []
    for r in rows:
        v = VERDICT[run][(r["model"], r["path"], r["offset_s"], r["timestamps"])]
        row = {k: r[k] for k in ("model", "path", "speaker", "region", "offset_s", "timestamps")}
        if TAX:
            row["taxonomy"], row["taxonomy_halluc"] = r["category"], r["taxonomy"]
        row.update(verdict=v["label"], halluc=r[run],
                   **{k: v[k] for k in ("finish_reason", "prompt_tokens", "cached_tokens",
                                        "completion_tokens", "reasoning_tokens",
                                        "response_model", "system_fingerprint", "batch")})
        out.append(row)
    for row in out:
        assert set(row) == set(SAFE_FIELDS), set(row) ^ set(SAFE_FIELDS)
        assert row["verdict"] in cfg["labels"], row["verdict"]
        for k, val in row.items():
            assert k in IDENT or " " not in str(val), (k, "free text in a numbers-only column")

    buf = io.StringIO()
    wr = csv.DictWriter(buf, fieldnames=SAFE_FIELDS, lineterminator="\n")
    wr.writeheader()
    wr.writerows(out)
    csv_path = os.path.join(RUNS_ROOT, run, "verdict_per_utterance.csv")
    csv_sha = write_bytes(csv_path, buf.getvalue().encode())
    back = list(csv.DictReader(open(csv_path, newline="")))
    assert len(back) == len(out) and back[0]["verdict"] == out[0]["verdict"]

    effort = cfg.get("reasoning_effort")
    deviations = ["response_format json_schema (strict) replaces the prompt's 'produce only the "
                  "classification' instruction, removing the parse step and foreclosing both a "
                  "preamble and an off-vocabulary label"]
    if effort is not None:
        deviations += [
            "no greedy decoding: the GPT-5 family rejects `temperature`, so the raw responses in "
            "this folder are the record of the run, not a re-derivable function of its inputs",
            f"reasoning_effort='{effort}'" + (
                ", matching their SSA.4.1 decision to avoid chain-of-thought generation"
                if effort == "none" else
                ", a deliberate departure from their SSA.4.1 no-chain-of-thought decision")]

    prov = {
        "experiment": "LLM-based hallucination classification (Atwany et al. 2025, ACL Findings) "
                      "over experiment A's hypotheses, as an independent check on the text taxonomy",
        "run": run,
        "prompt": "Atwany et al. Figure 5 (coarse-grained) / Figure 6 (fine), transcribed verbatim; "
                  "full text in run.json",
        "config": cfg,
        "hallucination_definition": {
            "counted_labels": sorted(COUNTED),
            "note": "applied to stored labels at analysis time. Atwany et al. count 'Hallucination "
                    "Error' only (their HER is that label's share, kept per cell under "
                    "summary.*.label_counts); 'Oscillation Error' is also counted here, following "
                    "Jasinski et al., who treat looping as a hallucination subtype"},
        "deviations": deviations,
        "licensing": rec["licensing"],
        "inputs": {**rec["inputs"], **({"halluc_taxonomy.csv": TAX_SHA} if TAX else {})},
        "packages": {"at_submission": rec.get("packages"),
                     "at_analysis": {"python": platform.python_version(), "numpy": np.__version__}},
        "batches": [{k: v for k, v in b.items() if k != "rows"} for b in rec["batches"]],
        "parse": PARSE[run],
        "summary": {k: v for k, v in SUMMARY.items() if k.split("|")[0] in ("taxonomy", run)},
        "agreement": {k: v for k, v in AGREE.items() if run in k.split(" vs ")},
        "outputs": {"verdict_per_utterance.csv": {"rows": len(out), "fields": SAFE_FIELDS,
                                                  "sha256": csv_sha, "git_safe": True}},
        "derived_at": now(),
    }
    write_json(os.path.join(RUNS_ROOT, run, "provenance.json"), prov)
    print(f"{run}\n  verdict_per_utterance.csv  {len(out)} rows, labels and numbers only\n"
          f"  provenance.json            rates, label counts, "
          f"${PARSE[run]['realised_cost_usd']:.2f} realised")

## 11. Figure

Hallucination rate at the headline offset against checkpoint, one panel per timestamp arm — the flat
off-arm is what makes the on-arm meaningful. Every run in `ANALYSE` is a series, and the taxonomy is
one more when its file was found.

Drawn in the project's house style from `Figures.ipynb`: no titles (inset `(a)`/`(b)` labels
instead), no parameter counts, axis labels of at most three words, a closed box with a light grid,
STIX serif, and a colour *and* dash pattern per series so it survives greyscale. Vector PDF at
`pdf.fonttype = 42`, verified below by inflating the PDF's streams, then downloaded with the PNG.

In [ ]:
import re, zlib
import matplotlib as mpl
import matplotlib.pyplot as plt

WIDTH, HEIGHT = 6.30, 2.35                 # ACL full width, inches -- never rescale in LaTeX
INK, RULE, GRIDC, MUTE = "#111111", "#3a3a3a", "#d2d2ce", "#9a9a95"
PALETTE = ["#000000", "#2a6fd6", "#1a9850", "#d1342f", "#7b5ea7"]
DASHES  = ["-", (0, (4.5, 1.6)), (0, (4, 1.4, 1, 1.4)), (0, (1.3, 1.3)),
           (0, (6, 1.4, 1, 1.4, 1, 1.4))]
MARKERS = ["o", "s", "^", "D", "v"]

mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "serif", "font.serif": ["STIXGeneral", "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7,
    "axes.spines.top": True, "axes.spines.right": True,
    "axes.edgecolor": RULE, "axes.linewidth": 0.7, "axes.labelcolor": INK,
    "axes.grid": True, "axes.axisbelow": True,
    "grid.color": GRIDC, "grid.linewidth": 0.45, "grid.linestyle": "-",
    "xtick.color": INK, "ytick.color": INK, "xtick.major.width": 0.7, "ytick.major.width": 0.7,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "lines.linewidth": 1.25, "lines.markersize": 3.2,
    "legend.frameon": False, "legend.handlelength": 2.6,
    "figure.dpi": 200, "figure.facecolor": "white",
    "savefig.bbox": "tight", "savefig.pad_inches": 0.02, "savefig.facecolor": "white",
})


def label_of(src):
    if src == "taxonomy":
        return "text taxonomy"
    cfg = RUNREC[src]["config"]
    how = "greedy" if cfg["reasoning_effort"] is None else f"effort {cfg['reasoning_effort']}"
    grains = {RUNREC[r]["config"]["grain"] for r in ANALYSE}
    return f"{cfg['judge']}, {how}" + (f", {cfg['grain']}" if len(grains) > 1 else "")


x = np.arange(len(MODELS))
dodge = (np.arange(len(SOURCES)) - (len(SOURCES) - 1) / 2) * 0.09
top = max(100 * v["wilson_hi"] for k, v in SUMMARY.items() if k.split("|")[2] == str(HEAD[0]))

fig, axes = plt.subplots(1, 2, figsize=(WIDTH, HEIGHT), sharey=True)
for ax, arm, tag in zip(axes, ("on", "off"), ("(a) timestamps on", "(b) timestamps off")):
    for n, src in enumerate(SOURCES):
        s   = [SUMMARY[f"{src}|{m}|{HEAD[0]}|{arm}"] for m in MODELS]
        mid = np.array([100 * v["rate"] for v in s])
        err = np.array([[100 * (v["rate"] - v["wilson_lo"]) for v in s],
                        [100 * (v["wilson_hi"] - v["rate"]) for v in s]])
        ax.errorbar(x + dodge[n], mid, yerr=err, fmt="none", ecolor=MUTE,
                    elinewidth=0.6, capsize=1.6, zorder=2)
        ax.plot(x + dodge[n], mid, color=PALETTE[n % 5], linestyle=DASHES[n % 5],
                marker=MARKERS[n % 5], markeredgecolor="white", markeredgewidth=0.4,
                zorder=3, label=label_of(src))
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS)
    ax.set_xlim(-0.5, len(MODELS) - 0.5)
    ax.text(0.972, 0.955, tag, transform=ax.transAxes, ha="right", va="top", color=INK, fontsize=8)
axes[0].set_ylim(0, max(1.0, top) * 1.3)
axes[0].set_ylabel("hallucination rate (%)")
axes[1].legend(loc="upper left")
fig.subplots_adjust(wspace=0.05)


def pdf_blobs(path):
    """matplotlib Flate-compresses its streams, so a byte grep for /Type3 checks nothing."""
    raw = open(path, "rb").read()
    out = [raw]
    for m in re.finditer(rb"stream\r?\n", raw):
        e = raw.find(b"endstream", m.end())
        try:
            out.append(zlib.decompress(raw[m.end():e]))
        except zlib.error:
            pass
    return b"\n".join(out)


STEM = "llm_halluc_rate"
for d in dict.fromkeys([".", RUNS_ROOT]):          # working dir, plus a copy beside the runs
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(d, f"{STEM}.{ext}"))
body = pdf_blobs(f"{STEM}.pdf")
assert b"/Type3" not in body and b"/Subtype /Image" not in body, "PDF is not clean vector"
assert b"/FontFile2" in body, "TrueType fonts not embedded: pdf.fonttype = 42 did not take effect"
print(f"wrote {STEM}.pdf (vector, TrueType verified) and .png, with copies in {RUNS_ROOT}")
plt.show()

if ON_COLAB:
    from google.colab import files
    files.download(f"{STEM}.pdf")
    files.download(f"{STEM}.png")

## 12. Standalone reload

No API key, no TIMIT, no raw files, no earlier cell — not even §1. Reads each run's git-safe
`verdict_per_utterance.csv` and rebuilds the hallucination-rate table (and agreement, when there is
more than one source), which proves those files alone carry the whole result. The rate uses the
`halluc` column as §10 wrote it; `verdict` holds the full label for any other definition.

In [ ]:
# --- standalone: run this alone in a fresh CPU runtime ---------------------------------
import csv, collections, itertools, math, os
import numpy as np

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_ = "/content/drive/MyDrive/NAACL/llm_runs"
except ImportError:
    ROOT_ = os.path.join(os.environ.get("NAACL_DATA")
                         or ("data" if os.path.isdir("data") else "."), "llm_runs")

RUNS_ = sorted(d for d in os.listdir(ROOT_)
               if os.path.exists(os.path.join(ROOT_, d, "verdict_per_utterance.csv")))
assert RUNS_, f"no verdict_per_utterance.csv under {ROOT_}: run section 10 first"
MODELS_ = ["tiny", "base", "small", "medium", "large-v3"]
CONDS_  = [(5, "on"), (5, "off"), (25, "on"), (25, "off")]

flags, cell_of = collections.defaultdict(dict), {}
for run in RUNS_:
    for r in csv.DictReader(open(os.path.join(ROOT_, run, "verdict_per_utterance.csv"), newline="")):
        key = (r["model"], r["path"], int(r["offset_s"]), r["timestamps"])
        if "taxonomy_halluc" in r:
            flags["taxonomy"][key] = int(r["taxonomy_halluc"])
        flags[run][key] = int(r["halluc"])
        cell_of[key] = key[0], key[2], key[3]

SRC_ = (["taxonomy"] if flags["taxonomy"] else []) + RUNS_
keys = sorted(cell_of)
assert all(set(flags[s]) == set(keys) for s in SRC_), "runs cover different rows"


def wilson_(k, n, z=1.959963985):
    p, d = k / n, 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


by_cell = collections.defaultdict(list)
for key in keys:
    by_cell[cell_of[key]].append(key)

print(f"reloaded {len(keys)} rows | runs: {', '.join(RUNS_)}\n")
for src in SRC_:
    print(src)
    print(f"{'model':>9} " + " ".join(f"{f'{o} s / ts {t}':>21}" for o, t in CONDS_))
    for m in MODELS_:
        out = []
        for o, t in CONDS_:
            ks = by_cell[(m, o, t)]
            p, lo, hi = wilson_(sum(flags[src][k] for k in ks), len(ks))
            out.append(f"{100*p:5.1f} [{100*lo:4.1f},{100*hi:4.1f}]")
        print(f"{m:>9} " + " ".join(f"{c:>21}" for c in out))
    print()

print("agreement (raw / kappa / jaccard)")
for sa, sb in itertools.combinations(SRC_, 2):
    a = np.array([flags[sa][k] for k in keys])
    b = np.array([flags[sb][k] for k in keys])
    po = float((a == b).mean())
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    kap = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    jac = (a & b).sum() / max(1, (a | b).sum())
    print(f"  {sa} vs {sb}: {po:.3f} / {kap:.3f} / {jac:.3f}")